In [ ]:
import sys
import os
import pickle
from typing import NamedTuple
import math
import numpy as np
import scipy as sp
import pandas as pd
from matplotlib import pyplot as plt


sys.path.append(os.path.join(os.getcwd(), '../../src/'))
from utils_mouse import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)

In [ ]:
ll = 'll00'
data_variant = 'gcamp'
model_variant = 'ns'
organism = 'fly-2023-schaffer'

In [ ]:
file_id = 2
neuron_labels = ['02_6463', '02_306', '02_6848', '02_8896', '02_7812']

In [ ]:
manual_reg_lambdas = {
	'wb_reg_lambda': 1E-1,
	'wm_reg_lambda': 1E-9,
}

with open('../../src/hyperparams.pkl', 'rb') as file:
    swept_params = pickle.load(file)
swept_reg_lambdas = dict()
for model in swept_params[ll].keys():
	type_, variant = model.split('_')
	if variant not in swept_reg_lambdas.keys():
		swept_reg_lambdas[variant] = dict()
	if type_ not in swept_reg_lambdas[variant].keys():
		swept_reg_lambdas[variant][type_] = dict()
	for param in swept_params[ll][model]['means'].keys():
		swept_reg_lambdas[variant][type_][param] = swept_params[ll][model]['means'][param]['geometric']


reg_lambdas = swept_reg_lambdas.copy()

bom_folder = (f'{data_variant}_bom_{model_variant}_'
			  + '_'.join([f'{param.split('_')[0]}{reg_lambdas[model_variant]['bom'][param]:.2E}' for param in reg_lambdas[model_variant]['bom'].keys()]))
hgm_folder = (f'{data_variant}_hgm_{model_variant}_'
			  + '_'.join([f'{param.split('_')[0]}{reg_lambdas[model_variant]['hgm'][param]:.2E}' for param in reg_lambdas[model_variant]['hgm'].keys()]))

In [ ]:
path = os.path.join(read_utils.get_data_path(), '6163622/sessions')

(filename,
 traces,
 continuous_behaviors,
 motion_pcs,
 motion_masks) = read_utils.get_data(path, file_id, with_whisking=True)

# clear nan values
# mask = motion_pcs[0].notna().all(axis=1)
mask = continuous_behaviors.notna().all(axis=1)

traces = traces[mask]
motion_pcs[0] = motion_pcs[0][mask]
motion_pcs[1] = motion_pcs[1][mask]
continuous_behaviors = continuous_behaviors[mask]

# naive ID unknown neurons
traces.columns = pd.Index([
	f'{file_id:02d}_{n_id:03d}' if empty or type(label) is not str else label for n_id, (label, empty) in
	enumerate(zip(traces.columns, traces.columns == ''))
])

# preprocess
def reconstruct_calcium(spikes, fs, decay=2.0):
	dt = 1.0 / fs
	gamma = np.exp(-dt / decay)
	trace = np.zeros(spikes.shape, dtype=np.float32)
	for t in range(1, spikes.shape[0]):
		trace[t, :] = gamma * trace[t - 1, :] + spikes[t, :]
	return trace

sampling_rate = read_utils.get_sampling_rate(os.path.join(path, filename))
traces = pd.DataFrame(reconstruct_calcium(traces.values, fs=sampling_rate), columns=traces.columns)
classes = pd.DataFrame(
	continuous_behaviors[['running_speed']].values > np.quantile(continuous_behaviors[['running_speed']].values, 0.05),
	columns=['run']
)

In [ ]:
_, bom_man, bom_rec, bom_res, bom_pres, _, _ = train_utils.load_predictions(os.path.join(read_utils.get_model_path(), 'predictions', bom_folder), filename)
_, hgm_man, hgm_rec, hgm_res, hgm_pres, _, _ = train_utils.load_predictions(os.path.join(read_utils.get_model_path(), 'predictions', hgm_folder), filename)

In [ ]:
splits = 5
plot_utils.plot_manifolds([f'{str(file_id)}: {filename} s{s}' for s in range(splits)],
						  [hgm_man[s] for s in range(splits)],
						  [classes for s in range(splits)],
						  None, None,
						  title='manifold', plots_per_row=splits, top_space=3.5,
						  save_path=None)

In [ ]:
splits = 5
plot_utils.plot_traces_splits(
	[hgm_rec[i][neuron_labels] for i in range(splits)],
	[bom_rec[i][neuron_labels] for i in range(splits)],
	traces[neuron_labels],
	classes[['run']], label='running',
	title=f'{filename} gcamp hgm & bom definitions manifold', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_MAN'

In [ ]:
splits = 5
plot_utils.plot_traces_splits(
	[hgm_res[i][neuron_labels] for i in range(splits)],
	[bom_res[i][neuron_labels] for i in range(splits)],
	traces[neuron_labels],
	classes[['run']], label='running',
	title=f'{filename} gcamp hgm & bom definitions residuals', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_RES'

In [ ]:
# max_x = traces.values.shape[0]
max_x = 1000

splits = 5
for i in range(splits):
	plot_utils.plot_traces_splits_pairs(
		[hgm_res[i][neuron_labels].iloc[:max_x, :],
		 hgm_pres[i][neuron_labels].iloc[:max_x, :]],
		[bom_res[i][neuron_labels].iloc[:max_x, :],
		 bom_pres[i][neuron_labels].iloc[:max_x, :]],
		traces[neuron_labels].iloc[:max_x, :],
		classes[['run']].iloc[:max_x, :], label='running',
		title=f'{filename} split {i} gcamp hgm & bom predictions residuals', ylimits=None, plots_per_row=1,
		save_path=None)  #../../out/plots/f'GCAMP_PRES_S{i}'

In [ ]:
sigmoid = lambda x: 1 / (1 + np.exp(-x))

neuron = neuron_labels[0]
neuron_id = traces.columns.get_loc(neuron)

splits = 5
sigmoid_terms = list()
for split in range(splits):
	manifold_term = hgm_man[split].values
	manifold_term_zs = (manifold_term - manifold_term.mean(axis=0)) / np.where((N_std := manifold_term.std(axis=0)) == 0.0, 1.0, N_std)

	WM = hgms_params[split]['model']['WM'][:, neuron_id]

	sigmoid_term = sigmoid(manifold_term_zs @ WM)
	sigmoid_terms.append(pd.DataFrame(data=sigmoid_term))

plot_utils.plot_manifolds_gradient([f'{str(file_id)}: {filename} s{s}' for s in range(splits)],
						  [hgm_man[s] for s in range(splits)],
						  [sigmoid_terms[s] for s in range(splits)],
						  None, None,
						  title='gcamp manifolds by gating', plots_per_row=splits, top_space=1,
						  save_path=None)